# Notebook 3 - $\color{green}\text{Knickpoint Identification}$


---

### Download anaconda3 and create a new conda environment using the following commands in the anaconda prompt:
```
cd <the directory this file and the yaml file is in>
conda env create -f ksn_env.yml
```

### Select the new environment by clicking 'select kernel' in the upper right.

In [1]:
import os, lsdtopytools as lsd, glob
from pathlib import Path

x:\anaconda3\envs\ksn_env\lib\site-packages\lsdtopytools\geoconvtools.py:79: NumbaDeprecationWarning: The keyword argument 'nopython=False' was supplied. From Numba 0.59.0 the default is being changed to True and use of 'nopython=False' will raise a warning as the argument will have no effect. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @nb.jit(nopython = False)
x:\anaconda3\envs\ksn_env\lib\site-packages\lsdtopytools\numba_tools.py:73: NumbaDeprecationWarning: The keyword argument 'nopython=False' was supplied. From Numba 0.59.0 the default is being changed to True and use of 'nopython=False' will raise a warning as the argument will have no effect. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @nb.jit(nopython=False, parallel = True)


In [9]:
# User defined parameters. Run this cell each time you update one of the enclosed variables.
PROJ_TITLE = 'example_directory' # Create a succinct name with no spaces or leading digits to represent your project file for future exports.

PATH = Path.cwd() # Gets the user's current working directory to avoid the use of absolute paths.

***

In [10]:
def ksn(wshed_list:list):
    '''
    This function uses LSDTopyTools' workflow to identify knickpoints and assign K_{sn} values for each pixel along the streams within a watershed.

    args: list of watershed paths.

    returns: none.
    '''
    # Here we use Simon Mudd's LSDTopyTools to identify knickpoints in each watershed.  More details on his work in the LSDTopotools documentation as well
    # as within the paper linked to in notebook 1.
    for path in wshed_list:
        name = path.name
        name_short = name[:-5]
        reproj_dir = Path(PATH) / PROJ_TITLE / 'reproj'
        mydem = lsd.LSDDEM(str(reproj_dir)+'/', name)
        print('Raster Loaded')
        mydem.PreProcessing()
        print('PreProcessed')
        mydem.CommonFlowRoutines()
        print('Flow routines got')
        mydem.ExtractRiverNetwork()
        print('River Network Extracted')
        mydem.DefineCatchment(method='main_basin')
        print('Catchment defined')
        mydem.GenerateChi()
        print('Chi generated')
        mydem.ksn_MuddEtAl2014(target_nodes=70, n_iterations=60, skip=1, minimum_segment_length=10, sigma=2,  nthreads = 1, reload_if_same = False)
        ksn_df = mydem.df_ksn
        mydem.knickpoint_extraction(lambda_TVD = "auto", combining_window = 30, window_stepped = 80, n_std_dev = 7)
        knickpoint_df = mydem.df_knickpoint
        knickpoint_df.to_csv(Path(kp_path) / f'{name_short}_knickpoints.csv')
        ksn_df.to_csv(Path(ksn_path) / f'{name_short}_ksn.csv')

# Hey!
## This next cell takes a very long time to work.  Get yourself some cookies and milk and wait it out.

In [11]:
dem_dir=Path(PATH) / PROJ_TITLE / 'reproj'
paths=list(dem_dir.glob('*.tiff'))

try:
    ksn_path = Path(PATH) / PROJ_TITLE / 'ksn_csvs'
    ksn_path.mkdir()
    kp_path = Path(PATH) / PROJ_TITLE / 'kp_csvs'
    kp_path.mkdir()
except(FileExistsError):
    print('File exists')
    kp_path = Path(PATH) / PROJ_TITLE / 'kp_csvs'
    ksn_path = Path(PATH) / PROJ_TITLE / 'ksn_csvs'
    
ksn(paths)

Loading the raster from file: d:\thesis_data_dict\notebooks_blank\example_directory\reproj/Big_Sandy_Creek_reproj.tiff
LOADING TOOK 0.05049777030944824
I am recasting your nodata values to -9999 (standard LSDTT)
PREPROC TOOK 0.0
Alright, let me summon control upon the c++ code ...
Got it.
INGESTINGINTO CPP TOOK 0.0034835338592529297
FINALISATION TOOK 0.25539517402648926
lsdtopytools is now ready to roll!
Raster Loaded
Carving: implementation of Lindsay (2016) DOI: 10.1002/hyp.10648
Filling: implementation of Wang and Liu (2006): https://doi.org/10.1080/13658810500433453
Processing...
DEM ready for flow routines!
PreProcessed
Processing common flow routines...
Done!
Flow routines got
River Network Extracted
Catchment defined
Chi generated
I have generated ksn for the specified region!
Let me just save the result to the hdf5 file to keep track
I got your knickpoints!
Loading the raster from file: d:\thesis_data_dict\notebooks_blank\example_directory\reproj/Coxes_Creek_reproj.tiff
LOADING